# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [36]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [37]:
df['revenue']=df['qty']*df['price']

total_revenue=df['revenue'].sum()
total_units=df['qty'].sum()

print("Total Revenue:", total_revenue)
print("Total Units:", total_units)



Total Revenue: 8520.0
Total Units: 783


In [38]:
#The total revenue from all of the items sold was $8520, and the total number of units sold, across all vendor and categories was 783.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [39]:
revenue_sums=df.groupby('category')['revenue'].sum()
revenue_sums=revenue_sums.sort_values(ascending=False)
revenue_sums=revenue_sums.to_frame()
revenue_sums['percent_total']=(revenue_sums['revenue']/total_revenue*100)

revenue_sums



,revenue,percent_total
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


In [40]:
#This table displays the total revenue for all of the items sold under specific categories. It also displays the percent of total revenue that each category makes up. We can see that food items produced the revenue, making up roughly half of the total.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [41]:
avg_and_count=df.groupby('vendor_id')['revenue'].agg(['mean','count'])
avg_and_count=avg_and_count.sort_values('mean', ascending=False)
avg_and_count


,mean,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


In [42]:
#This table displays the average order revenue for each vendor and the number of each orders has vendor has achieved. We can see that vendor V-01 has the highest average order revenue. All of the vendor haves order counts in a similar range.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [43]:
merch_revenue=df[df['category']=='Merch']['revenue'].sum()
merch_share=(merch_revenue/total_revenue)*100
print("Share of Revenue from Merch:", round(merch_share, 1))



Share of Revenue from Merch: 20.8


In [44]:
#Rougly 21% percent of revenue comes from merch, making it the second highest revenue category, after food.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [45]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

rows_before= len(df)
revenue_before= df['revenue'].sum()

merged_df=df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

rows_after=len(merged_df)
revenue_after=merged_df['revenue'].sum()

print("Row Count Same:", rows_before == rows_after)
print("Revenue Same:", revenue_before == revenue_after)

unmatched_vendor=merged_df[merged_df['vendor_name'].isna()]['vendor_id'].unique()

print("Unmatched vendor:", unmatched_vendor)

merged_df['vendor name']= merged_df['vendor_name'].fillna('Unknown vendor')
# TODO: merge, validate, and report the unmatched vendor



Row Count Same: True
Revenue Same: True
Unmatched vendor: ['V-18']


In [46]:
#By joining the two tables, it is now possible to match the vendor name to the different vendor ids. The checks confirmed that after merging the two tables, no rows or revenue were lost.

**The unmatched vendor, and what I did about it: The unmatched vendor is V-18. I didn't want to lose the rows or revenue for this vendor, so I just assigned it the name of "Unknown vendor" using the .fillna() function.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [47]:
table=pd.pivot_table(
    merged_df,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    observed=True,
    margins=True,
    margins_name='Total'
)

table



category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Total,972.0,3274.5,1263.0,661.5,6171.0


In [48]:
#This table clearly lays out vendors and their revenue in each of four categories, as well as their total revenue across those categories. We can see that Cav Merch North has the highest total revenue, with Hoos Burger following just slightly behind.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [49]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_sums['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged_df) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) There are a couple of recommendations I'd make to the vendors to maximize revenue. For one, I'd highlight that rain gear only made up ~10% of revenue. Given this low percentage, vendors should take into account the weather forecast and consider reducing rain gear inventory. In contrast to the recommendation for rain gear, I'd advise vendors to increase and diversify their food inventory, given that it makes up half of all revenue.

b) I believe the revenue by category analysis to be somewhat surface level/untrustworthy. The categories are quite broad, so it's impossible to know what specific products are producing revenue. For example, we know that food is driving half of revenue, but we don't know what specific food products are selling.